In [ ]:
import requests
import tqdm
import os
import io
import concurrent.futures
from PIL import Image
import argparse
import nltk
nltk.download('punkt', quiet=True)
import pytesseract
from py2opsin import py2opsin
from pathlib import Path
home = Path.home()
data_path = home / "projects/Markush/data"

OLLAMA_URL = "http://localhost:11434/api/generate"


def ocr_page(page_data):
    """
    OCR function that works with ProcessPoolExecutor.
    Takes serialized image data.
    """
    page_index, img_bytes = page_data
    img = Image.open(io.BytesIO(img_bytes))
    text = pytesseract.image_to_string(img)
    return page_index, text


def ocr_pages(doc, dpi=300, workers=4):
    """
    Perform OCR on all pages of a PDF document.
    """
    if doc.page_count == 0:
        return [], ""
    ocr_texts = [""] * doc.page_count

    # Pre-serialize pages for ProcessPoolExecutor
    page_data_list = []
    for i in range(doc.page_count):
        page = doc.load_page(i)
        pix = page.get_pixmap(dpi=dpi, alpha=False)
        img_bytes = pix.tobytes("png")
        page_data_list.append((i, img_bytes))

    # Perform OCR on all pages in parallel
    with concurrent.futures.ProcessPoolExecutor(max_workers=min(os.cpu_count(), len(page_data_list), workers)) as executor:
        futures = {executor.submit(ocr_page, p): p[0] for p in page_data_list}
        for future in concurrent.futures.as_completed(futures):
            page_index, text = future.result()
            ocr_texts[page_index] = text

    return ocr_texts


def extract_iupac_using_llm(text_input: str, model_name: str) -> str:
    prompt = f"Extract the IUPAC names from the following text. \
    Output the IUPAC names only without any other description. \
    Return each name on a separate line. If there is no IUPAC name, return ''\
    .\nText: {text_input}"
    payload = {
        "model": model_name,
        "prompt": prompt,
        "stream": False
    }
    response = requests.post(OLLAMA_URL, json=payload)
    response.raise_for_status()
    result = response.json()
    iupac_name = result.get("response", "").strip()
    return iupac_name if iupac_name else ""


def iupac_to_smile(iupac_names: list) -> list[tuple[str, str]]:
    compounds = []
    for name in iupac_names:
        try:
            smile = py2opsin(name)
            if smile:
                compounds.append((name, smile))
        except Exception as e:
            print(f"Error converting {name} to SMILES: {e}")
    return compounds


def main(year: int, patent_no_list: list[int] = None):
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_name", type=str, required=True, help="Ollama model name")
    args = parser.parse_args()
    model_name = args.model_name

    Path.mkdir(data_path / f'{year}/compounds', parents=True, exist_ok=True)
    if patent_no_list is None:
        pdf_files = list((data_path / str(year) / 'pages_with_markush').glob("*.pdf"))
    else:
        pdf_files = [data_path / str(year) / 'pages_with_markush' / f'{str(patent_no)}.pdf' for patent_no in patent_no_list]





    result_dir = "data/paper_html_10.1038/abs_annotation/generated_annotations" # automatically generating output file paths based on model used
    output_filename = f"{model_name+ f"_{abstract_type}" if abstract_type != "full" else model_name}.txt"
    OUTPUT_PATH = os.path.join(result_dir, output_filename)

    os.makedirs(result_dir, exist_ok=True)

    with open(OUTPUT_PATH, "a", encoding="utf-8") as f_out:
        for i in tqdm.tqdm(range(start_index, len(test_abstracts)), desc="Generating TLDR", 
                           total=len(test_abstracts)-start_index, unit="annotation"):
            payload = {
                "model": model_name,
                "prompt": test_abstracts[i],
                "stream": False
            }

            response = requests.post(OLLAMA_URL, json=payload)
            response.raise_for_status()
            result = response.json()
            generated_tldr = result.get("response", "") or ""

            filtered_tldr_single_line = " ".join(filtered_tldr.split())
            f_out.write(filtered_tldr_single_line + "\n")
            f_out.flush()

if __name__ == "__main__":
    main()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

./tokenizer.model:   0%|          | 0.00/1.48M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

AttributeError: 'NoneType' object has no attribute 'shape'